# Multi-Spectrum Line Analysis

In this notebook, we demonstrate how to pack multiple spectra into a Data1D container, fit them simultaneously using `MultiFit`, and extract array-valued metrics (like Equivalent Width) natively.

In [5]:
import numpy as np
import astropy.units as u
from prism.data import Spectrum, Data1D
from prism.modeling.models.lines import GaussianLines, setup_local_lines
from astropy.modeling.models import Polynomial1D
from prism.modeling.fitting import TRFLSQFitter

## 1. Simulate 5 Spectra
We'll create 5 mock spectra with varying line amplitudes.

In [11]:
wave = np.linspace(4800, 5100, 300) * u.AA
spectra = []

# Base model
cont = Polynomial1D(c0=10.0, degree=0)
lines = GaussianLines.from_arrays(
    names=['Hb', 'OIII'],
    pos=[4861.333, 5006.843],
    position_unit=u.AA,
    fwhm=300.0,
    amplitude=[5.0, 10.0]
)
base_model = cont + lines

np.random.seed(42)
for i in range(5):
    # Vary the OIII amplitude
    model_i = base_model.copy()
    model_i.right.amp_oiii.value = 10.0 + i * 2.0
    
    flux = model_i(wave)
    noise = np.random.normal(0, 0.5, size=flux.size)
    
    spec = Spectrum(flux + noise, x=wave, unit=u.Jy)
    spec.variance = np.full_like(flux, 0.5**2)
    spectra.append(spec)


ValueError: shape mismatch: objects cannot be broadcast to a single shape.  Mismatch is between arg 0 with shape (300,) and arg 1 with shape (2,).

## 2. Pack into Data1D and MultiFit
We stack the spectra to perform a parallel `MultiFit`.

In [ ]:
# Stack spectra into a single Data1D object
data = Data1D.from_spectra(spectra)

# Set up the fitter and MultiFit
fitter = TRFLSQFitter()
multifit = MultiFit(base_model, fitter)

# Run the fit across all 5 spaxels
result = multifit.fit(data)
print("Fit completed.")


## 3. Extract Array-Valued Equivalent Width
Using the `.ew()` method on the component view, we can extract the Equivalent Width for all spectra simultaneously.

In [ ]:
# Get the line group component view
lines_view = result.get_component('GaussianLines')

# Calculate Equivalent Width using the continuum component
ew_array = lines_view.ew(continuum=result.get_component('Polynomial1D').evaluate())

print("EW values across 5 spectra:")
print(ew_array.value)


## 4. Bisector Span via Line Analysis
We can also use `SelectedLineCollection` to select a specific line and run Monte Carlo measurements (like bisector spans) on all spectra.

In [ ]:
# Select the OIII line across the entire MultiFit result
oiii_collection = SelectedLineCollection(result, 'OIII')

# Measure the bisector span (velocity asymmetry) for each spectrum
for i in range(data.n_spaxels):
    profile = oiii_collection.get_profile(i)
    measurements = profile.measure(x=wave)
    print(f"Spectrum {i+1} Bisector Span: {measurements['bisector_span'].value:.2f} km/s")
